In [36]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
df = pd.read_csv("AB_NYC_2019.csv")
print(df.shape)
df.info()
df.head()

(48895, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last_review         

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [37]:
# Missing values and Duplicates
print(df.isna().sum()) 
print("duplicates:", df.duplicated().sum())

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64
duplicates: 0


In [42]:
# Handle Missing Values for each column in name, host_name, 



# For empty last_review, 



# 4. Clean up the temporary column
# df.drop(columns=['name_filled'], inplace=True)

# miss = df.reviews_per_month.isna()
# print(miss.sum(), (df.number_of_reviews[miss] == 0).all())

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,name_filled


In [5]:
print((df.price == 0).sum(), "listings with price 0")
print(df.price.describe())
print(df.price.quantile([.9, .95, .99, .999]))
print((df.minimum_nights > 365).sum(), "listings with minimum_nights > 365")

11 listings with price 0
count    48895.000000
mean       152.720687
std        240.154170
min          0.000000
25%         69.000000
50%        106.000000
75%        175.000000
max      10000.000000
Name: price, dtype: float64
0.900     269.0
0.950     355.0
0.990     799.0
0.999    3000.0
Name: price, dtype: float64
14 listings with minimum_nights > 365


In [44]:
n0 = len(df)

# Since name, host_name, last_review couldn't be derived and not critical for our analysis, 
# ignoring these null values

# 1. Missing reviews: flag first, then fill with 0
df["has_reviews"] = df.reviews_per_month.notna().astype(int)
df["reviews_per_month"] = df.reviews_per_month.fillna(0)

# 2. Remove listings with price <= 0
valid_price = df.price > 0
price_below_99_percentile = df.price <= df.price.quantile(.99)
d = df[valid_price & price_below_99_percentile]
print(f"Removed zero prices and outliers {n0:.0f}:", n0 - len(d))

# 3. Remove minimum nights > 365 [Not Short-term rentals]
n2 = len(d)
d = d[d.minimum_nights <= 365]
print("Removed min nights > 365:", n2 - len(d))

# 5. Log of price for modeling, then save
d = d.copy()
d["log_price"] = np.log(d.price)
print("final rows:", len(d))
d.to_csv("AB_NYC_2019_clean.csv", index=False)
d.head()

Removed zero prices and outliers 48895: 485
Removed min nights > 365: 14
final rows: 48396


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,name_filled,has_reviews,log_price
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365,Clean & quiet apt home by the park,1,5.003946
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355,Skylit Midtown Castle,1,5.416100
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,0.00,1,365,THE VILLAGE OF HARLEM....NEW YORK !,0,5.010635
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194,Cozy Entire Floor of Brownstone,1,4.488636
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0,Entire Apt: Spacious Studio/Loft by central park,1,4.382027
